# Steps a2/a3: recognising the piece from short windows

- **a2**: feature baselines (tempo, dundun pattern, jembe pattern, jembe timing, jembe feel without pattern) with logistic regression, random forest and nearest centroid. CPU, a few minutes.
- **a3**: a small 1D CNN (11k parameters) and a small transformer (68k parameters) on stroke sequences, same leave-one-take-out folds and metrics. Use a GPU runtime (T4).

Corpus: IEMP Malian Jembe, CC BY-NC-ND 4.0, downloaded into the session only. Metrics and figures go to `MyDrive/djembe/recognition/`; no model weights are saved by this notebook.

## 1. Code and pinned dependencies
If pip reports that numpy was replaced, use *Runtime > Restart session*, then continue from section 2.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Djembe" if IN_COLAB else os.path.abspath("..")
COMMIT = "ee54db53252ba9ced53dbaaa6a668598725d8b68"   # pinned code version; bump it when the code changes
if IN_COLAB and not os.path.exists(REPO):
    !git clone -q https://github.com/clementdumeril/Djembe.git "{REPO}" && git -C "{REPO}" checkout -q {COMMIT}
%pip install -q -r "{REPO}/requirements-train.txt"

## 2. Device and output folder

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Djembe" if IN_COLAB else os.path.abspath("..")
sys.path.insert(0, REPO); os.chdir(REPO)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    DEVICE = "cpu"
print("device:", DEVICE)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/djembe/recognition"
    CORPUS = "/content/corpus"
else:
    OUT_DIR = os.path.join(REPO, "output", "recognition")
    CORPUS = os.path.join(REPO, "corpus")
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["DJEMBE_CORPUS"] = CORPUS  # read by tests/test_splits.py
print("outputs ->", OUT_DIR)

## 3. Annotations and event table

In [ ]:
import djembe_dataset as d
d.download_corpus(CORPUS)
events_path = os.path.join(OUT_DIR, "events.csv")
d.build_events(CORPUS).to_csv(events_path, index=False)
!python -m pytest -q tests

## 4. a2: feature baselines

In [ ]:
a2_json = os.path.join(OUT_DIR, "a2_recognition.json")
a2_fig = os.path.join(OUT_DIR, "a2_window_accuracy.png")
!python -W ignore recognition_baselines.py --events "{events_path}" --out "{a2_json}" --figure "{a2_fig}"
from IPython.display import Image
Image(a2_fig)

## 5. a3: sequence models
About 1 to 3 minutes per configuration on a T4. Three seeds per configuration; the folds are the same as in a2.

In [ ]:
a3_json = os.path.join(OUT_DIR, "a3_sequence.json")
!python sequence_models.py --events "{events_path}" --out "{a3_json}" --windows 1 2 4 8 --seeds 0 1 2

## 6. Side by side

In [ ]:
import json, pandas as pd
a2 = json.load(open(a2_json))["results"]; a3 = json.load(open(a3_json))["results"]
rows = []
for k in a3:
    for fs in ("d1_pattern", "jembe_pattern", "jembe_feel3"):
        rows.append({"k": int(k), "model": f"logreg {fs}", "bal_acc": a2[k][fs]["logistic_regression"]["balanced_accuracy"]})
    for voice in a3[k]:
        for name, r in a3[k][voice].items():
            rows.append({"k": int(k), "model": f"{name} {voice}", "bal_acc": r["balanced_accuracy_mean"]})
pd.DataFrame(rows).pivot(index="model", columns="k", values="bal_acc").round(3)